# Phase 2 — Data Cleaning: Validation & Verification

**Status:** ✅ Completed (pipeline v2.0)

This notebook is a **read-only validation layer** over `ml_service/src/clean.py`.
It loads `artifacts/cleaning_summary.json` and the cleaned CSV files to verify:
- Before vs. after row counts and removal accounting
- Zero NaN, zero inf, zero duplicates in all cleaned files
- Label normalization audit
- Column schema decomposition (76 + 1 + 1 + 1 = 79)
- Class distribution before and after cleaning
- Source integrity SHA-256 digests

> **Do not perform any data modifications in this notebook.**
> The authoritative cleaning implementation is `ml_service/src/clean.py`.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.ARTIFACTS_DIR / 'cleaning_summary.json', 'r', encoding='utf-8') as f:
    cs = json.load(f)

print(f"Phase:            {cs['phase']}")
print(f"Pipeline version: {cs['pipeline_version']}")
print(f"Timestamp (UTC):  {cs['timestamp_utc']}")
print(f"Elapsed:          {cs['elapsed_seconds']} s")
print(f"Peak RSS:         {cs['memory']['peak_rss_mb']} MB  (limit: {cs['memory']['memory_limit_mb']} MB, within_limit={cs['memory']['within_limit']})")

## 1. Global Accounting — Before vs. After

In [ ]:
ga = cs['global_accounting']
d = ga['duplicate_rows_removed']

rows = [
    ('Rows Read (2,830,743 original flows)', ga['rows_read']),
    ('Removed — NaN / Infinity', ga['rows_with_nan_or_inf_removed']),
    ('Removed — Within-File Duplicates', d['within_file']),
    ('Removed — Cross-File Duplicates', d['cross_file']),
    ('Total Removed', ga['rows_with_nan_or_inf_removed'] + d['total']),
    ('Rows Written (Clean Unique Flows)', ga['rows_written']),
]
pd.DataFrame(rows, columns=['Metric', 'Count']).style.format({'Count': '{:,}'})

## 2. Per-File Cleaning Summary

In [ ]:
records = []
for fr in cs['per_file_reports']:
    d = fr['duplicate_rows_removed']
    records.append({
        'File': fr['file_name'],
        'Rows Read': fr['rows_read'],
        'NaN/Inf Removed': fr['rows_with_nan_or_inf_removed'],
        'Within-File Dupes': d['within_file'],
        'Cross-File Dupes': d['cross_file'],
        'Rows Written': fr['rows_written'],
    })
df_files = pd.DataFrame(records)
df_files.style.format({c: '{:,}' for c in df_files.columns if c != 'File'})

## 3. Label Normalizations Applied

In [ ]:
norm = cs.get('global_label_normalizations', {})
if norm:
    pd.DataFrame(list(norm.items()), columns=['Transformation', 'Occurrences'])
else:
    print('✅ No label normalizations were required.')

## 4. Column Schema (76 + 1 + 1 + 1 = 79)

In [ ]:
with open(Paths.ARTIFACTS_DIR / 'column_schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)

print(f"Total columns:              {schema['total_columns']}")
print(f"ML feature columns:         {schema['ml_feature_columns_count']}")
print(f"Network identifier columns: {schema['identifier_columns']}")
print(f"Redundant columns:          {schema['redundant_columns']}")
print(f"Target column:              {schema['target_column']}")
print()
print(f"Identity check: {schema['ml_feature_columns_count']} + {len(schema['identifier_columns'])} + {len(schema['redundant_columns'])} + 1 = {schema['total_columns']}")

## 5. Independent Verification Results

In [ ]:
v = cs['post_cleaning_verification']
print(f"Overall: {'✅ ALL PASSED' if v['all_passed'] else '❌ FAILURES FOUND'}")
print()
pd.DataFrame(v['per_file_verification'])

## 6. Class Distribution in Cleaned Dataset

In [ ]:
label_counts = {}
total_clean = 0

for f in sorted(Paths.CLEANED_DIR.glob('*_cleaned.csv')):
    for chunk in pd.read_csv(f, usecols=['Label'], chunksize=50_000):
        for label, count in chunk['Label'].value_counts().items():
            label_counts[label] = label_counts.get(label, 0) + int(count)
            total_clean += int(count)

df_labels = pd.DataFrame(list(label_counts.items()), columns=['Label', 'Count'])
df_labels['Percentage (%)'] = (df_labels['Count'] / total_clean * 100).round(4)
df_labels.sort_values('Count', ascending=False).reset_index(drop=True)

## 7. Source Integrity — Raw CSVs Unchanged

In [ ]:
integrity = cs['source_integrity']
rows = [
    {'File': k, 'Rows': v['rows'], 'Columns': v['columns'], 'SHA-256 (first 32)': v['sha256'][:32] + '...'}
    for k, v in integrity.items()
]
pd.DataFrame(rows)